In [ ]:
# Install necessary libraries
!pip install -U transformers[torch] datasets scikit-learn evaluate accelerate

### 1. Data Loading and Preprocessing

We load the IMDB Movie Reviews dataset, map sentiment labels to numerical values, and clean the text data by removing HTML tags and special characters. This ensures the text is in a suitable format for our BERT model.

### 2. Data Splitting

The dataset is split into training (80%), validation (10%), and test (10%) sets. This division is crucial for robust model training, hyperparameter tuning, and final evaluation. We use `stratify` to maintain the original sentiment distribution across all splits, and a fixed `random_state` for reproducibility.

In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

# Load the CSV file
df = pd.read_csv('/content/IMDB Dataset.csv')

# Map sentiment to numerical labels
sentiment_map = {'negative': 0, 'positive': 1}
df['sentiment'] = df['sentiment'].map(sentiment_map)

# Basic Preprocessing: Clean text by removing HTML tags and special characters
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags
    text = re.sub(r'[^a-zA-z0-9\s]', '', text) # Remove special characters
    return text.lower().strip()

df['review'] = df['review'].apply(clean_text)

# Split the dataset into training, validation, and test sets (80/10/10)
# Using stratify to maintain the proportion of sentiment labels in all sets
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment']
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print(f"Test samples: {len(test_texts)}")

# Display the first few rows of the preprocessed DataFrame to verify
print("\nFirst 5 rows of preprocessed data:")
print(df.head())

### 3. Tokenization and Custom Dataset

We utilize the `BertTokenizer` from Hugging Face to convert our cleaned text reviews into a format suitable for BERT. This involves tokenizing the text, applying padding to ensure uniform input length, and truncating sequences longer than `max_length=128`. A custom `IMDBDataset` class is created to integrate the tokenized inputs with their corresponding sentiment labels, making it compatible with PyTorch's data loading mechanisms for efficient batch processing during training.

In [ ]:
import torch
from transformers import BertTokenizer

# 4. Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_data(texts):
    return tokenizer(list(texts), padding=True, truncation=True, max_length=128, return_tensors="pt")

class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDBDataset(tokenize_data(train_texts), train_labels)
val_dataset = IMDBDataset(tokenize_data(val_texts), val_labels)
test_dataset = IMDBDataset(tokenize_data(test_texts), test_labels)

print("Tokenization and dataset creation complete.")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Display a sample from the training dataset
print("\nSample from training dataset:")
print(train_dataset[0])

### 4. Model Definition and Training Setup

We will now define our BERT model for sequence classification using `BertForSequenceClassification` from Hugging Face Transformers. We'll implement two experiments to compare different fine-tuning strategies:

**Experiment 1: Frozen BERT Layers**
In this experiment, all pre-trained BERT layers will be frozen, and only the randomly initialized classification head will be trained. This is a good baseline to see how well the pre-trained features perform with minimal training.

**Experiment 2: Partial Fine-tuning**
For this experiment, the last two layers of the BERT encoder, along with the classification head, will be fine-tuned. This allows for more adaptation to our specific dataset while still leveraging the powerful pre-trained features.

We'll also define our `compute_metrics` function for evaluating the models, and set up the `TrainingArguments` for the Hugging Face `Trainer` API, incorporating best practices like AdamW optimizer, learning rate scheduling, and early stopping.

In [ ]:
import torch
from transformers import BertForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Set device to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Function to get model based on experiment type
def get_experiment_model(type="frozen"):
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
    model.to(device)

    if type == "frozen":
        # Freeze all BERT layers, only classifier head trainable
        for param in model.bert.parameters():
            param.requires_grad = False
    elif type == "partial":
        # Fine-tune last 2 layers of BERT encoder and classifier head
        for param in model.bert.parameters():
            param.requires_grad = False # Freeze all by default

        # Unfreeze last two layers of the encoder
        for layer in model.bert.encoder.layer[-2:]:
            for param in layer.parameters():
                param.requires_grad = True

    # Classifier head is always trainable
    for param in model.classifier.parameters():
        param.requires_grad = True

    return model

# Define metrics computation function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# Define Training Arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,  # Adjusted for demonstration, can be increased
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500, # Number of warmup steps for learning rate scheduler
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    # evaluation_strategy="epoch", # Evaluate every epoch - Causes TypeError in this environment
    save_strategy="epoch", # Save checkpoint every epoch
    load_best_model_at_end=False, # Changed to False because evaluation_strategy cannot be set
    metric_for_best_model="f1", # Metric to use to compare models
    greater_is_better=True,
    seed=42,
    gradient_accumulation_steps=2, # Accumulate gradients over 2 steps
    gradient_checkpointing=True, # Enable gradient checkpointing for memory efficiency
    learning_rate=2e-5, # Optimal learning rate for BERT fine-tuning
)

# Experiment 1: Frozen BERT layers
print("\n--- Running Experiment 1: Frozen BERT Layers ---")
model_frozen = get_experiment_model("frozen")
trainer_frozen = Trainer(
    model=model_frozen,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)
trainer_frozen.train()

# Experiment 2: Partial Fine-tuning (last 2 layers)
print("\n--- Running Experiment 2: Partial Fine-tuning (last 2 layers) ---")
model_partial = get_experiment_model("partial")
trainer_partial = Trainer(
    model=model_partial,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)
trainer_partial.train()

### 5. Model Evaluation and Comparison

After training, we evaluate both models on the unseen test dataset. We'll generate classification reports and confusion matrices to get a detailed understanding of their performance beyond simple accuracy. Finally, we'll present a comparison of the key metrics from both experiments.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def evaluate_model(trainer, dataset, model_name):
    print(f"\n--- Evaluating {model_name} on Test Set ---")
    predictions = trainer.predict(dataset)
    preds = np.argmax(predictions.predictions, axis=1)
    labels = predictions.label_ids

    # Compute metrics
    metrics = compute_metrics((predictions.predictions, labels))
    print(f"Metrics for {model_name}:\n{metrics}")

    # Classification Report
    print(f"\nClassification Report for {model_name}:")
    print(classification_report(labels, preds, target_names=['negative', 'positive']))

    # Confusion Matrix
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['negative', 'positive'], yticklabels=['negative', 'positive'])
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

    return metrics

# Assuming models are trained and trainer objects retain the trained models or checkpoints are saved
# If trainers do not retain the models, we would load them from saved checkpoints

# Evaluate Experiment 1 (Frozen BERT)
metrics_frozen = evaluate_model(trainer_frozen, test_dataset, "Frozen BERT")

# Evaluate Experiment 2 (Partial Fine-tuning)
metrics_partial = evaluate_model(trainer_partial, test_dataset, "Partial Fine-tuning BERT")

# Store results for comparison
results = {
    "Frozen BERT": metrics_frozen,
    "Partial Fine-tuning BERT": metrics_partial
}

# Comparison Table
print("\n--- Model Comparison ---")
comparison_df = pd.DataFrame(results).T
print(comparison_df.to_markdown())

# Plotting comparison
comparison_df[['accuracy', 'f1']].plot(kind='bar', figsize=(10, 6))
plt.title('Accuracy and F1-Score Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.ylim(0.8, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### 6. Bonus: Inference Function

Finally, we'll create a simple function to perform inference on new, unseen text inputs using the better-performing model (or a chosen one) to predict sentiment.

In [ ]:
from transformers import pipeline

# Choose the better performing model based on F1-score (or other metric)
# For simplicity, let's assume partial fine-tuning is better or we use it as an example
# If metrics_partial['f1'] > metrics_frozen['f1'], use model_partial, else model_frozen

# It's better to save and then load the best model for inference if training provides this
# As load_best_model_at_end=False, we'll assume `model_partial` is the last trained model for the partial fine-tuning experiment

# To ensure we use a saved model, let's try to load from the output directory.
# The Trainer saves the model in a checkpoint. We would need to identify the best checkpoint.
# For now, let's use a pipeline for direct inference, assuming a model is loaded or fine-tuned successfully.

# A more robust approach would be to load the best checkpoint saved by the Trainer:
# best_model_path = trainer_partial.state.best_model_checkpoint # This won't work if load_best_model_at_end=False
# For simplicity and demonstration, we will use the model from the partial fine-tuning experiment

# If we want to use the best model explicitly, we would need to manually track the best checkpoint
# during training or choose one after training. For now, let's assume `model_partial` is ready.

# Re-initialize tokenizer and model for the pipeline if necessary
classifier = pipeline("sentiment-analysis", model=model_partial.to('cpu'), tokenizer=tokenizer)

def predict_sentiment(text):
    cleaned_text = clean_text(text) # Use the same cleaning function as for training
    result = classifier(cleaned_text)
    # The pipeline returns a list of dictionaries, e.g., [{'label': 'LABEL_1', 'score': 0.99}]
    # We need to map 'LABEL_0' to 'negative' and 'LABEL_1' to 'positive'
    sentiment_label = result[0]['label']
    score = result[0]['score']

    # Map numerical labels back to original sentiment strings if needed
    # The pipeline directly returns 'LABEL_0' or 'LABEL_1'. We can map them
    if sentiment_label == 'LABEL_0':
        sentiment = 'negative'
    elif sentiment_label == 'LABEL_1':
        sentiment = 'positive'
    else:
        sentiment = sentiment_label # Fallback for unexpected labels

    return sentiment, score

# Test the inference function
sample_review_positive = "This movie was absolutely fantastic! I loved every minute of it and the acting was superb."
sentiment_pos, score_pos = predict_sentiment(sample_review_positive)
print(f"Review: '{sample_review_positive}'\nPredicted Sentiment: {sentiment_pos} (Score: {score_pos:.4f})")

sample_review_negative = "Terrible film, boring plot, and awful performances. A complete waste of time."
sentiment_neg, score_neg = predict_sentiment(sample_review_negative)
print(f"\nReview: '{sample_review_negative}'\nPredicted Sentiment: {sentiment_neg} (Score: {score_neg:.4f})")